# Figures and Statistics for Thermal Landscapes Project - Upland/Lowland Comparisons
 PB 01/29/23
 
Reads in pickle object files made with RectifiedImageProcessing scripts, and makes plots for the Thermal Landscapes paper.

This script only uses sites with upland/lowland splits:
- 'RoanCamp'
- 'Letaba'
- 'Hlangwine'
- 'Nkuhlu'

In [9]:
from pathlib import Path
import xarray as xr
import rioxarray as rioxr
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import pickle
from scipy.stats import ttest_ind
import numpy as np


# Set figure directory for output figures
figdir = Path(f'./figs/final_120722/AllSites')
if not figdir.exists():
    figdir.mkdir()

# Pickle dir
pickledir = Path('./data/out/pixelpickles/')

# Random Number Seed
# Set a random number seed so that the analysis is reproducible 1/16/23
# https://stats.stackexchange.com/questions/354373/what-exactly-is-a-seed-in-a-random-number-generator
seed = 42

# # # END USER INPUTS

In [8]:
# Define Settings for Plotting

# Set the seaborn plot theme to default
# incorporates grey grid into background of figures
custom_params = {"axes.spines.right": False, "axes.spines.top": False}
sns.set_theme(font_scale=1.3, style="ticks", rc=custom_params)

# Set hue order for plotting
exclosure_hues=["Outside", "Inside"]
topo_hues = ["Upland", "Lowland"]

# List out Sites in Order (top to bottom) for plotting
sites = ['Letaba', 'Nkhulu', 'Hlangwine', 'RoanCamp']

In [4]:
# Open up pickles and save in dictionaries
pix_dict = {}

for p in pickledir.glob('*.obj'):
    if p.name.split('_')[0] in sites:
        df = pd.read_pickle(p)
        projstr = p.name.split('_')[0]
        pix_dict[projstr] = df.copy(deep=True)

In [10]:
# Define functions for stats and effect size calculation
# Added more stats 1/26/23

# Also, calc the number and proportion of images with significant p-values and mean diff < 0
# Aka - proporation of images with significant differences that showed inside being cooler than outside
def calcpropimages(ES):
    trues = [((es.p <= 0.001) & (es.meandiff < 0)) for es in ES.itertuples()]
    ntrues = np.sum(trues)
    prop = ntrues / len(trues)
    return prop

# Also, calc the number and proportion of images with significant p-values and mean diff < 0
# Aka - proporation of images with significant differences that showed inside being TALLER than Outside
def calcpropimages_PAI(ES):
    trues = [((es.p <= 0.001) & (es.meandiff > 0)) for es in ES.itertuples()]
    ntrues = np.sum(trues)
    prop = ntrues / len(trues)
    return prop

# Also, calc the number and proportion of images with significant p-values and mean diff < 0
# Aka - proporation of images with significant differences that showed inside being TALLER than Outside
def calcpropimages_height(ES):
    trues = [((es.p <= 0.001) & (es.meandiff > 0)) for es in ES.itertuples()]
    ntrues = np.sum(trues)
    prop = ntrues / len(trues)
    return prop

def EffectSize_by_Exclosure(df, var='Temperature', seed=None):

    # set random number seed
    if not seed:
        seed = 42
        
    # Split by Inside and Outside
    df_in = df.loc[df.Exclosure=='Inside']
    df_out = df.loc[df.Exclosure=='Outside']
    
    # Get sample size (number of pixels)
    n_in = len(df_in[var])
    n_out = len(df_out[var])
    
    # Find the smaller area
    # and set the number of pixels to sample 
    # to be ~66% of the number of pixels in the smallest area
    numpix = int(np.floor(0.66*np.min([n_in, n_out])))
            
    # Sample pixels to account for spatial autocorrelation
    # Added a random number seed here so that results are reproducible 1/16
    df_in = df_in.sample(n=numpix, random_state=seed)
    df_out = df_out.sample(n=numpix, random_state=seed)
    
    # Run a welch's t-test (equal variances not assumed)
    t, p = ttest_ind(df_in[var],
                     df_out[var],
                     equal_var=False)
    
    # Also, run Cohen's D as a measure of effect size per image
    cd = cohen_d(df_in[var],
                 df_out[var])
    
    # Get sample size (number of pixels), and means
    n_in_final = len(df_in[var])
    n_out_final = len(df_out[var])
    
    mean_in = df_in[var].mean()
    mean_out = df_out[var].mean()
    meandiff = mean_in-mean_out
    
    std_in = df_in[var].std()
    std_out = df_out[var].std()
    stddiff = df_in[var].std() - df_out[var].std()
    
    CV_in =  df_in[var].std() / df_in[var].mean()
    CV_out = df_out[var].std() / df_out[var].mean()
    CVdiff = (df_in[var].std() / df_in[var].mean()) - (df_out[var].std() / df_out[var].mean())
    
    # Compile results
    results = (t, p, cd,
               meandiff, mean_in, mean_out,
               n_in_final, n_out_final,
               stddiff, std_in, std_out,
               CVdiff, CV_in, CV_out)
    
    return results

# Cohen's D - A Measure of effect size 
# aka: a measure of standardized mean difference
# mean1 - mean2 / pooled sample Std
# https://stackoverflow.com/questions/21532471/how-to-calculate-cohens-d-in-python/33002123#33002123
# Other useful links:
# http://ethen8181.github.io/machine-learning/ab_tests/causal_inference/matching.html
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html
# https://en.wikipedia.org/wiki/Effect_size#Difference_family:_Effect_sizes_based_on_differences_between_means
def cohen_d(x, y):
    nx = len(x)
    ny = len(y)
    dof = nx + ny - 2
    return (np.mean(x) - np.mean(y)) / np.sqrt(((nx-1)*np.std(x, ddof=1) ** 2 + (ny-1)*np.std(y, ddof=1) ** 2) / dof)


In [11]:
# Loop through and calculate statistics, grouped by different variables (img, topo, and exclosure)
# Note that you sample 66% of the exlosure with the smallest number of pixels (inside or outside) 
# Each time you run it, it will resample those randomly (so it's not reproducable).

# Added an Upland/Lowland splitting step here
for topo in ['Upland', 'Lowland']:

    # initialize list for temperature ttest and effect size to make allsites file
    df_ES_list_temp = []
    df_ES_list_pai = []
    df_ES_list_height = []

    for s in sites:

        # Get all pixels included in the current site
        df_pixels = pix_dict[s]
        
        # FILTER for upland/lowland up top 
        df_pixels = df_pixels.loc[df_pixels['Topo'] == topo]

        # # # 1)  Make output summary stat files
        statsbyEx = df_pixels.groupby(['Exclosure', 'Topo']).describe()

        statsbyEx.to_csv(f'./stats/Stats_byExclosure_{s}-{topo}.csv')

        statsbyExbyImg = df_pixels.groupby(['imgf', 'Exclosure']).describe()

        statsbyExbyImg.to_csv(f'./stats/Stats_byExclosure_byImage_{s}-{topo}.csv')

        if 'Topo' in df_pixels.columns:

            statsbyExbyTopo = df_pixels.groupby(['Topo', 'Exclosure']).describe()

            statsbyExbyTopo.to_csv(f'./stats/Stats_byExclosure_byTopoPosition_{s}-{topo}.csv')

            statsbyExbyImgbyTopo = df_pixels.groupby(['imgf', 'Topo', 'Exclosure']).describe()

            statsbyExbyImgbyTopo.to_csv(f'./stats/Stats_byExclosure_byTopoPosition_byImage_{s}-{topo}.csv')

        # # # 2) Make effect size and ttest statistic files by image and summarized for each site
        statsbyImg = df_pixels.groupby(['imgf'])

        # # Ttest and cohens d for temperature
        ES_results = statsbyImg.apply(lambda x: EffectSize_by_Exclosure(x, var='Temperature', seed=seed))

        df_list = []

        for i in ES_results.iteritems():

            # Results Tuple is organized like:
            # (t, p, cd,
            #        meandiff, mean_in, mean_out,
            #        n_in_final, n_out_final,
            #        stddiff, std_in, std_out,
            #        CVdiff, CV_in, CV_out,)

            df = pd.DataFrame({'site':s,
                                 'imgf': [i[0]],
                                 'tstat':[i[1][0]],
                                 'p':[i[1][1]],
                                 'cohensd':[i[1][2]],
                                 'meandiff':[i[1][3]],
                                 'mean_inside':[i[1][4]],
                                 'mean_outside':[i[1][5]],
                                 'n_inside':[i[1][6]],
                                 'n_outside':[i[1][7]],
                                 'stddiff':[i[1][8]],
                                 'std_inside':[i[1][9]],
                                 'std_outside':[i[1][10]],
                                 'CVdiff':[i[1][11]],
                                 'CV_inside':[i[1][12]],
                                 'CV_outside':[i[1][13]]})

            df_list.append(df)

        # Compile temperature statistics for this site
        # and save
        ESstats_s = pd.concat(df_list, ignore_index=True)
        ESstats_s.to_csv(f'./stats/TTestandEffectSizeStats_Temperature_byExclosure_byImage_{s}-{topo}.csv')
        df_ES_list_temp.append(ESstats_s)

        # # Ttest and cohens d for Height
        ES_Height_results = statsbyImg.apply(lambda x: EffectSize_by_Exclosure(x, var='Height', seed=seed))

        df_height_list = []

        for i in ES_Height_results.iteritems():

            df = pd.DataFrame({'site':s,
                                 'imgf': [i[0]],
                                 'tstat':[i[1][0]],
                                 'p':[i[1][1]],
                                 'cohensd':[i[1][2]],
                                 'meandiff':[i[1][3]],
                                 'mean_inside':[i[1][4]],
                                 'mean_outside':[i[1][5]],
                                 'n_inside':[i[1][6]],
                                 'n_outside':[i[1][7]],
                                 'stddiff':[i[1][8]],
                                 'std_inside':[i[1][9]],
                                 'std_outside':[i[1][10]],
                                 'CVdiff':[i[1][11]],
                                 'CV_inside':[i[1][12]],
                                 'CV_outside':[i[1][13]]})

            df_height_list.append(df)

        # Compile temperature statistics for this site
        # and save
        ESstats_height_s = pd.concat(df_height_list, ignore_index=True)
        ESstats_height_s.to_csv(f'./stats/TTestandEffectSizeStats_Height_byExclosure_byImage_{s}-{topo}.csv')
        df_ES_list_height.append(ESstats_height_s)

        # # Ttest and cohens d for PAI
        ES_PAI_results = statsbyImg.apply(lambda x: EffectSize_by_Exclosure(x, var='PAI', seed=seed))

        df_PAI_list = []

        for i in ES_PAI_results.iteritems():

            df = pd.DataFrame({'site':s,
                                 'imgf': [i[0]],
                                 'tstat':[i[1][0]],
                                 'p':[i[1][1]],
                                 'cohensd':[i[1][2]],
                                 'meandiff':[i[1][3]],
                                 'mean_inside':[i[1][4]],
                                 'mean_outside':[i[1][5]],
                                 'n_inside':[i[1][6]],
                                 'n_outside':[i[1][7]],
                                 'stddiff':[i[1][8]],
                                 'std_inside':[i[1][9]],
                                 'std_outside':[i[1][10]],
                                 'CVdiff':[i[1][11]],
                                 'CV_inside':[i[1][12]],
                                 'CV_outside':[i[1][13]]})

            df_PAI_list.append(df)

        # Compile temperature statistics for this site
        # and save
        ESstats_PAI_s = pd.concat(df_PAI_list, ignore_index=True)
        ESstats_PAI_s.to_csv(f'./stats/TTestandEffectSizeStats_PAI_byExclosure_byImage_{s}-{topo}.csv')
        df_ES_list_pai.append(ESstats_PAI_s)
    
    # Start compiling stats
    # Compile ES temperature statistics from all sites
    ESstats_allsites = pd.concat(df_ES_list_temp, ignore_index=True)

    # save them 
    ESstats_allsites.to_csv(f'./stats/TTestandEffectSizeStats_Temperature_byExclosure_byImage_AllSites-{topo}.csv')

    # Group by site for below calculations
    ESstats_bysite = ESstats_allsites.groupby(['site'])

    # Redone 1/26 - export stats of all vars at once
    for c in ['tstat', 'p', 'cohensd', 'meandiff', 'mean_inside',
           'mean_outside', 'n_inside', 'n_outside', 'stddiff', 'std_inside',
           'std_outside', 'CVdiff', 'CV_inside', 'CV_outside']:

        ESstats_bysite[c].describe().to_csv(f'./stats/{c}_Temperature_Summary_AllSites-{topo}.csv')

    propimg_summary = ESstats_bysite.apply(lambda x: calcpropimages(x))
    propimg_summary.to_csv(f'./stats/PropImageswithSigDiffinTemperature_AllSites-{topo}.csv')
    
    
    # Compile ES Height statistics from all sites
    ESstats_height_allsites = pd.concat(df_ES_list_height, ignore_index=True)

    # save them 
    ESstats_height_allsites.to_csv(f'./stats/TTestandEffectSizeStats_Height_byExclosure_byImage_AllSites-{topo}.csv')

    # Group by site for below calculations
    ESstats_height_bysite = ESstats_height_allsites.groupby(['site'])


    # Redone 1/26 - export stats of all vars at once
    for c in ['tstat', 'p', 'cohensd', 'meandiff', 'mean_inside',
           'mean_outside', 'n_inside', 'n_outside', 'stddiff', 'std_inside',
           'std_outside', 'CVdiff', 'CV_inside', 'CV_outside']:

        ESstats_height_bysite[c].describe().to_csv(f'./stats/{c}_Height_Summary_AllSites-{topo}.csv')
        
    propimg_summary_height = ESstats_height_bysite.apply(lambda x: calcpropimages_height(x))
    propimg_summary_height.to_csv(f'./stats/PropImageswithSigDiffinHeight_AllSites-{topo}.csv')

    
    # Compile ES PAI statistics from all sites
    ESstats_PAI_allsites = pd.concat(df_ES_list_pai, ignore_index=True)

    # save them 
    ESstats_PAI_allsites.to_csv(f'./stats/TTestandEffectSizeStats_PAI_byExclosure_byImage_AllSites-{topo}.csv')

    # Group by site for below calculations
    ESstats_PAI_bysite = ESstats_PAI_allsites.groupby(['site'])

    # Redone 1/26 - export stats of all vars at once
    for c in ['tstat', 'p', 'cohensd', 'meandiff', 'mean_inside',
           'mean_outside', 'n_inside', 'n_outside', 'stddiff', 'std_inside',
           'std_outside', 'CVdiff', 'CV_inside', 'CV_outside']:

        ESstats_PAI_bysite[c].describe().to_csv(f'./stats/{c}_PAI_Summary_AllSites-{topo}.csv')

    propimg_summary_PAI = ESstats_PAI_bysite.apply(lambda x: calcpropimages_PAI(x))
    propimg_summary_PAI.to_csv(f'./stats/PropImageswithSigDiffinPAI_AllSites-{topo}.csv')